### Imports and env

In [1]:
%run import_local_packages.py

import os

import random
import numpy as np
import torch
import yaml
from dotenv import load_dotenv
import wandb

from src.models.adapters.models.classifier import Classifier # Import all models into the registry
from src.models.adapters.vision_backbones import * # Import all backbones into the registry
from src.models.core.services import MODEL_REGISTRY
from src.benchmark.classification.full_shot.training_utils.trainers import ClassificationTrainer
from src.data.splits.mvtec import MVTech_SP_split
from src.benchmark.classification.full_shot.configs.training.base_train_config import TrainConfig
from src.benchmark.classification.full_shot.training_utils.loss import ClassificationLoss
from src.benchmark.classification.full_shot.utils import write_results_to_csv, get_hardware_config, mean_and_std_of_dicts, init_wandb, create_datasets, create_loaders

/Users/theo.moreau/Documents/futur/.pixi/envs/default/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
load_dotenv()

TRAIN_CONFIG_PATH = "./configs/training"
MODEL_CONFIG_PATH = "./configs/models"
RESULTS_CSV_PATH = "./results.csv"
CHECKPOINT_SAVE_PATH = "./checkpoints"

LOG_TO_WANDB = int(os.getenv("LOG_TO_WANDB"))
IMAGE_PATH = os.getenv("IMAGE_PATH")
HARDWARE_CONFIG_PATH = os.getenv("HARDWARE_CONFIG_PATH")

In [4]:
tracker, hw_config = get_hardware_config(HARDWARE_CONFIG_PATH)

[codecarbon INFO @ 08:45:40] [setup] RAM Tracking...
[codecarbon INFO @ 08:45:40] [setup] GPU Tracking...
[codecarbon INFO @ 08:45:40] No GPU found.
[codecarbon INFO @ 08:45:40] [setup] CPU Tracking...
[codecarbon WARNING @ 08:45:40] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon WARNING @ 08:45:41] We saw that you have a Apple M4 but we don't know it. Please contact us.
[codecarbon INFO @ 08:45:41] CPU Model on constant consumption mode: Apple M4
[codecarbon INFO @ 08:45:41] >>> Tracker's metadata:
[codecarbon INFO @ 08:45:41]   Platform system: macOS-15.5-arm64-arm-64bit
[codecarbon INFO @ 08:45:41]   Python version: 3.11.13
[codecarbon INFO @ 08:45:41]   CodeCarbon version: 2.2.2
[codecarbon INFO @ 08:45:41]   Available RAM : 24.000 GB
[codecarbon INFO @ 08:45:41]   CPU count: 10
[codecarbon INFO @ 08:45:41]   CPU model: Apple M4
[codecarbon INFO @ 08:45:41]   GPU count: None
[codecarbon INFO @ 08:45:41]   GPU model: None


Config already exists at: /Users/theo.moreau/Documents/futur/src/benchmark/hardware_configs/Apple M4_24.0GB.yaml


### Configure training

In [5]:
FOLDS_NUMBER = 2 # Number of folds for cross-validation
SAMPLES_PER_DEFECT = 100 # Maximum number of samples per defect class
TRAIN_KEEP_RATIO = 1.0 # Percentage of training samples to keep

In [6]:
training_configs = ["augmentation"] # Training configurations to launch

# Models your want to train
model_configs = [
    #"hresnet50_ft_linear_224",
    #"dino_s_ft_linear_224", 
    #"swin_t_ft_linear_224",
    "efficientnet_s_ft_linear_224",
    #"pe_frz_linear_224",
    #"vitb_frz_linear_224"
]

### Training loop

In [7]:
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

In [8]:
for training_config in training_configs:
    for model_config in model_configs:

        with open(os.path.join(TRAIN_CONFIG_PATH, training_config + ".yaml"), "r") as f:
            training_cfg = yaml.safe_load(f)

        with open(os.path.join(MODEL_CONFIG_PATH, model_config + ".yaml"), "r") as f:
            model_cfg = yaml.safe_load(f)
        
        training_cfg = TrainConfig(**training_cfg["training"])
        classes = training_cfg.classname
        model_cfg["num_classes"] = training_cfg.nb_classes
        config_name = model_config + "_" + training_config
        output_filename = config_name + ".pth"

        loss = ClassificationLoss(training_cfg.criterion, training_cfg.loss_weights, device)

        classes_final_results = {}
        for c in classes:
            splitter = MVTech_SP_split(
                dataset_path=os.path.join(IMAGE_PATH, training_cfg.dataset_name), 
                classname=c,
                train_split=training_cfg.train_split,
                multiclass=training_cfg.multiclass,
                samples_per_defect=SAMPLES_PER_DEFECT,
                total_folds=FOLDS_NUMBER,
                train_keep_ratio=TRAIN_KEEP_RATIO
            )
            
            #splitter.plot_dist()

            folds = splitter.supervised_train_test()
            final_results = []
            for idx in range(FOLDS_NUMBER):
                print("Training on fold ", idx)
                model = MODEL_REGISTRY.build_model(model_cfg)
                train_dataset, val_dataset, noisy_val_dataset = create_datasets(folds[idx], model.transform, training_cfg)
                train_loader, val_loader, noisy_val_loader = create_loaders(train_dataset, val_dataset, noisy_val_dataset, training_cfg)
                optimizer = torch.optim.Adam(
                    [
                        {"params": model.get_backbone_parameters(), "lr": training_cfg.lr[1]},
                        {"params": model.get_classifier_parameters(), "lr": training_cfg.lr[0]}
                    ],
                    weight_decay=training_cfg.weight_decay
                )
                trainer = ClassificationTrainer(optimizer, loss, training_cfg.train_metrics, training_cfg.val_metrics)
                
                if LOG_TO_WANDB:
                    run, artifact = init_wandb(c, SAMPLES_PER_DEFECT, TRAIN_KEEP_RATIO, training_cfg, model_cfg)
                else:
                    run = None
                    artifact = None
                
                save_path = os.path.join(CHECKPOINT_SAVE_PATH, output_filename)

                trainer.train(
                    model=model,
                    epochs=training_cfg.num_epochs,
                    train_dataloader=train_loader, 
                    validation_dataloader=val_loader, 
                    nb_classes=training_cfg.nb_classes,
                    device=device,
                    metric_steps=5,
                    save_path=save_path,
                    wandb_run=run,
                    model_artifact=artifact
                )
                
                results = trainer.validate_model(model, val_loader, noisy_val_loader, training_cfg.nb_classes, device, run, save_path=save_path)
                if LOG_TO_WANDB:
                    wandb.finish()
                final_results.append(results)
            classes_final_results[f"{c}"] = mean_and_std_of_dicts(final_results)
        for c in classes:
            write_results_to_csv(config_name, hw_config, c, SAMPLES_PER_DEFECT, TRAIN_KEEP_RATIO, classes_final_results, len(train_dataset), csv_path=RESULTS_CSV_PATH)

Training on fold  0


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/theo.moreau/.netrc
wandb: Currently logged in as: theo-moreau (pep23) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Train epoch 0: 100%|██████████| 18/18 [00:28<00:00,  1.60s/Batch]


train loss: 0.6978783077663846


Computing train metrics: 100%|██████████| 1/1 [00:00<00:00, 32.43Metric/s]


...

Train epoch 69: 100%|██████████| 18/18 [00:33<00:00,  1.87s/Batch]


train loss: 0.06474613994659092


Computing test metrics: 100%|██████████| 2/2 [00:00<00:00, 28.48Metric/s]


FPS measurement: 100%|██████████| 50/50 [00:04<00:00, 12.45it/s]
Unsupported operator aten::silu_ encountered 102 time(s)
Unsupported operator aten::add_ encountered 35 time(s)
Unsupported operator aten::sigmoid encountered 30 time(s)
Unsupported operator aten::mul encountered 30 time(s)
The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
backbone.model.features.1.0.stochastic_depth, backbone.model.features.1.1.stochastic_depth, backbone.model.features.2.0.stochastic_depth, backbone.model.features.2.1.stochastic_depth, backbone.model.features.2.2.stochastic_depth, backbone.model.features.2.3.stochastic_depth, backbone.model.features.3.0.stochastic_depth, backbone.model.features.3.1.stochastic_depth, backbone.model.featur

avg_train_loss,█▇▅▄▄▃▃▃▃▃▁▁▁▁
avg_val_loss,█▇▄▃▂▂▂▂▂▂▂▁▁▁
epoch,▁▂▂▃▃▄▄▅▅▆▆▇▇█
train_AP,▁▃▆▇▇█████████
train_loss,▃▃▃▃▃▃▃▂▂▁▃▂▁█▁▁▂▂▂▁▂▁▁▁▁▁▂▁▁▄▁▁▁▁▁▂▁▁▁▂
val_AP,▁▂▂▃▃▅▆▆▇▆▆▇▇█
val_expected_gain,▁▃▄▅▅▅▆▇█▆▆▆▇█
val_optimal_threshold,▅▆▆▇▃▃▇▁▇█▂▁▄▅
FLOPS,25337123328
FPS,62.479
avg_train_loss,0.10872
